# Global Model Comparison Analysis

This notebook analyzes the differences between three global models:
- `wind_100ex50` (100 stations)
- `wind_50` (50 stations)
- `wind_50_nostatic` (50 stations without static features)

## Analysis Steps:
1. Load daily R² CSV files for each model
2. Calculate pairwise differences and average
3. Reshape and merge metrics for comparison
4. Visualize specific forecast runs with interactive plots

In [10]:
import pandas as pd
import numpy as np
import plotly.io as pio
import plotly.graph_objects as go
from pathlib import Path
import copy
import torch
import yaml
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import gc
import os

# Import project utilities
from utils.models import get_model
from utils import preprocessing, tools, hpo

## 1. Load Daily R² CSV Files

Update these paths according to your actual model file names:

In [2]:
# Define paths to CSV files
data_dir = Path('data/test_results')

# TODO: Update these with your actual model identifiers
csv_100 = data_dir / 'daily_results_wind_100.csv'  # Adjust filename
csv_50 = data_dir / 'daily_results_wind_50.csv'  # Adjust filename
csv_50_nostatic = data_dir / 'daily_results_wind_50_nostatic.csv'  # Adjust filename
csv_local = data_dir / 'daily_results_local.csv'  # Adjust filename
csv_wind = 'data/wind_speed_eval/r2_per_forecast_run.csv'

# Load CSVs
df_100 = pd.read_csv(csv_100, index_col=0, parse_dates=True)
df_50 = pd.read_csv(csv_50, index_col=0, parse_dates=True)
df_50_nostatic = pd.read_csv(csv_50_nostatic, index_col=0, parse_dates=True)
df_local = pd.read_csv(csv_local, index_col=0, parse_dates=True)
df_wind = pd.read_csv(csv_wind, index_col=0, parse_dates=True)

print(f"df_100 shape: {df_100.shape}")
print(f"df_50 shape: {df_50.shape}")
print(f"df_50_nostatic shape: {df_50_nostatic.shape}")
print(f"df_local shape: {df_local.shape}")
print(f"df_wind shape: {df_wind.shape}")

# Show first few rows
df_100.head()

df_100 shape: (364, 100)
df_50 shape: (364, 50)
df_50_nostatic shape: (364, 50)
df_local shape: (364, 100)
df_wind shape: (364, 100)


,00164,00183,00232,00342,00433,00704,00867,00880,00963,01200,...,05779,05906,05930,06163,07370,07394,07403,15000,15444,15813
forecast_date,,,,,,,,,,,,,,,,,,,,,
2025-08-01 06:00:00+00:00,0.120618,-0.082121,-2.858552,0.484798,-3.301459,-0.579164,0.537165,-0.497729,-0.336256,0.083695,...,0.369124,0.214127,0.380570,0.084771,0.156195,0.467217,-2.475296,0.242766,-1.732553,0.101710
2025-08-01 09:00:00+00:00,-0.589899,0.418625,-0.890075,0.520137,-3.113962,-0.599395,-0.364158,0.280959,-0.331383,0.479198,...,-0.646274,0.159189,0.603364,-0.045292,-0.281958,0.015604,-0.166654,0.464660,-0.240230,-0.085743
2025-08-01 12:00:00+00:00,0.171657,0.276239,-0.174721,0.204252,-0.405765,-0.656382,-0.288903,0.288439,-0.894088,0.127463,...,-1.336346,0.262478,0.265080,-0.295835,-0.194707,0.153625,-1.390990,0.457705,0.352780,0.237459
2025-08-01 15:00:00+00:00,0.630011,0.255404,-0.015356,0.049787,0.216215,-1.320868,-0.256506,-0.023892,-0.589700,0.265300,...,-1.378516,0.070490,0.349951,0.142508,0.522085,0.437649,-0.896992,0.499638,0.333397,0.362401
2025-08-02 06:00:00+00:00,0.388171,0.361532,0.050845,0.272647,-0.876625,-2.296317,-0.157609,0.145421,-0.042400,-0.800516,...,0.248456,0.304082,0.460343,-0.589820,-1.827483,0.595511,-3.117723,0.586213,0.294356,0.444454


## 2. Calculate Pairwise Differences

Calculate absolute differences between models and average them:

In [3]:
# Calculate pairwise absolute differences
diff_100_50 = np.abs(df_100 - df_50)
diff_50_nostatic = np.abs(df_50 - df_50_nostatic)
diff_100_nostatic = np.abs(df_100 - df_50_nostatic)
diff_100_local = np.abs(df_100 - df_local)

# Calculate mean of the three differences
df_diff = (diff_100_50 + diff_50_nostatic + diff_100_nostatic + diff_100_local) / 4

print(f"df_diff shape: {df_diff.shape}")
print(f"\nMean difference across all forecasts/stations: {df_diff.mean().mean():.4f}")
print(f"Max difference: {df_diff.max().max():.4f}")
print(f"Min difference: {df_diff.min().min():.4f}")

df_diff.head()

df_diff shape: (364, 100)

Mean difference across all forecasts/stations: 2.1803
Max difference: 4596.3722
Min difference: 0.0000


,00164,00183,00198,00232,00282,00298,00303,00342,00427,00433,...,07393,07394,07395,07403,07410,07412,13674,15000,15444,15813
forecast_date,,,,,,,,,,,,,,,,,,,,,
2025-08-01 06:00:00+00:00,NaN,NaN,0.213251,NaN,0.232079,0.104222,0.155070,NaN,0.263973,NaN,...,NaN,0.080835,NaN,1.247846,NaN,NaN,NaN,0.048201,1.456334,0.115914
2025-08-01 09:00:00+00:00,NaN,NaN,0.255382,NaN,0.142686,0.064399,0.149122,NaN,0.212657,NaN,...,NaN,0.153291,NaN,0.324631,NaN,NaN,NaN,0.112932,0.428146,0.193488
2025-08-01 12:00:00+00:00,NaN,NaN,0.024198,NaN,0.090635,0.057936,0.589536,NaN,0.113263,NaN,...,NaN,0.191976,NaN,0.774913,NaN,NaN,NaN,0.084739,0.088738,0.156459
2025-08-01 15:00:00+00:00,NaN,NaN,0.044492,NaN,0.122496,0.038886,0.343168,NaN,0.102927,NaN,...,NaN,0.113396,NaN,0.666981,NaN,NaN,NaN,0.056862,0.140002,0.108561
2025-08-02 06:00:00+00:00,NaN,NaN,0.050365,NaN,0.052893,0.091880,0.892875,NaN,0.451532,NaN,...,NaN,0.160872,NaN,1.434579,NaN,NaN,NaN,0.059053,0.141865,0.163499


## 3. Reshape with Melt and Merge Metrics

Transform from wide to long format and add model metrics:

In [4]:
# Melt df_diff to long format
df_diff_melted = df_diff.reset_index().melt(
    id_vars='forecast_date',
    var_name='station_id',
    value_name='diff'
)

print(f"df_diff_melted shape: {df_diff_melted.shape}")
df_diff_melted.head()

df_diff_melted shape: (36400, 3)


,forecast_date,station_id,diff
0,2025-08-01 06:00:00+00:00,00164,NaN
1,2025-08-01 09:00:00+00:00,00164,NaN
2,2025-08-01 12:00:00+00:00,00164,NaN
3,2025-08-01 15:00:00+00:00,00164,NaN
4,2025-08-02 06:00:00+00:00,00164,NaN


In [5]:
# Melt the three model DataFrames
df_100_melted = df_100.reset_index().melt(
    id_vars='forecast_date',
    var_name='station_id',
    value_name='metrics_100'
)

df_50_melted = df_50.reset_index().melt(
    id_vars='forecast_date',
    var_name='station_id',
    value_name='metrics_50'
)

df_50_nostatic_melted = df_50_nostatic.reset_index().melt(
    id_vars='forecast_date',
    var_name='station_id',
    value_name='metrics_50_nostatic'
)

df_local_melted = df_local.reset_index().melt(
    id_vars='forecast_date',
    var_name='station_id',
    value_name='metrics_local'
)

df_wind_melted = df_wind.reset_index().melt(
    id_vars='forecast_date',
    var_name='station_id',
    value_name='metrics_wind'
)

# Merge all metrics with df_diff_melted
df_comparison = df_diff_melted.copy()
df_comparison = df_comparison.merge(
    df_100_melted[['forecast_date', 'station_id', 'metrics_100']],
    on=['forecast_date', 'station_id'],
    how='left'
)
df_comparison = df_comparison.merge(
    df_50_melted[['forecast_date', 'station_id', 'metrics_50']],
    on=['forecast_date', 'station_id'],
    how='left'
)
df_comparison = df_comparison.merge(
    df_50_nostatic_melted[['forecast_date', 'station_id', 'metrics_50_nostatic']],
    on=['forecast_date', 'station_id'],
    how='left'
)

df_comparison = df_comparison.merge(
    df_local_melted[['forecast_date', 'station_id', 'metrics_local']],
    on=['forecast_date', 'station_id'],
    how='left'
)

df_comparison = df_comparison.merge(
    df_wind_melted[['forecast_date', 'station_id', 'metrics_wind']],
    on=['forecast_date', 'station_id'],
    how='left'
)

# Sort by metrics_100
df_comparison = df_comparison.sort_values('metrics_100', ascending=False)

print(f"df_comparison shape: {df_comparison.shape}")
print(f"\nColumns: {df_comparison.columns.tolist()}")

# Display results
df_comparison.head(20)

df_comparison.to_csv('data/test_results/daily_model_comparison.csv', index=False)

df_comparison shape: (36400, 8)

Columns: ['forecast_date', 'station_id', 'diff', 'metrics_100', 'metrics_50', 'metrics_50_nostatic', 'metrics_local', 'metrics_wind']


## 3.1 Correlation Wind NWP Forecat Performance and Model Performance

In [53]:
df_comparison.dropna()[['metrics_100', 'metrics_wind']].corr()

,metrics_100,metrics_wind
metrics_100,1.000000,0.176779
metrics_wind,0.176779,1.000000


## 4. Analyze High-Difference Forecast Runs

Find forecast runs where models differed most:

In [6]:
# Top 20 forecast runs with highest differences
df_highest_diff = df_comparison.sort_values('diff', ascending=False).head(20)

print("Top 20 forecast runs with highest model differences:")
df_highest_diff[['forecast_date', 'station_id', 'diff', 'metrics_100', 'metrics_50', 'metrics_50_nostatic', 'metrics_local']]

Top 20 forecast runs with highest model differences:


,forecast_date,station_id,diff,metrics_100,metrics_50,metrics_50_nostatic,metrics_local
4585,2025-09-24 09:00:00+00:00,00596,4596.372213,-3671.640842,-4113.757630,-11350.807583,-6698.796211
11508,2025-09-26 06:00:00+00:00,01694,2360.363856,-4225.541633,-2607.753190,-1853.059090,-8922.031971
30068,2025-09-25 06:00:00+00:00,05930,2085.721875,-681.553880,-217.123986,-2979.749594,-3499.190164
11509,2025-09-26 09:00:00+00:00,01694,1508.536741,-1503.642126,-1566.880906,-1210.002524,-6824.032323
30069,2025-09-25 09:00:00+00:00,05930,1473.129210,-354.176128,-84.473529,-2591.193707,-1233.252612
30071,2025-09-25 15:00:00+00:00,05930,1389.453140,-1987.183069,-92.482107,-1692.797825,-218.772431
18772,2025-09-22 06:00:00+00:00,03158,1228.003196,-2666.720027,-928.866138,-1799.514514,-1230.415020
11510,2025-09-26 12:00:00+00:00,01694,1180.269959,-880.251703,-775.734270,-591.097346,-5023.022823
4586,2025-09-24 12:00:00+00:00,00596,1160.513793,-1212.523970,-1611.300932,-3368.661288,-1542.304504
17989,2025-09-08 09:00:00+00:00,03098,1158.807238,-1043.973784,-1109.998044,-461.104193,-4381.415033


## 5. Interactive Forecast Visualization

Plot predictions from all three models for a specific forecast run.

### 5.1 Configuration

In [6]:
# Configuration for forecast visualization
# Define the forecast run and station to visualize first
forecast_run = '2025-10-25 15:00:00+00:00'  # Update with desired timestamp
station_id_to_plot = '05546'  # Update with desired station ID

# Model paths for global models
model_paths = {
    '100': Path('models/cl_m-tft_out-48_freq-1h_wind_100.pt'),
    '50': Path('models/cl_m-tft_out-48_freq-1h_wind_50.pt'),
    '50_nostatic': Path('models/cl_m-tft_out-48_freq-1h_wind_50_nostatic.pt'),
    'local': None  # Will be set dynamically based on station_id_to_plot
}

# Set local model path based on station ID
# Local models are named like: cl_m-tft_out-48_freq-1h_<station_id>.pt
local_model_pattern = f'cl_m-tft_out-48_freq-1h_wind_{station_id_to_plot}.pt'
local_model_path = Path('models') / local_model_pattern

if local_model_path.exists():
    model_paths['local'] = local_model_path
    print(f"✓ Found local model: {local_model_path.name}")
else:
    print(f"⚠ Warning: Local model not found: {local_model_path}")
    print(f"  Continuing without local model")

# Config path (used for all models)
config_path = Path('configs/config_wind_100.yaml')

print(f"\nForecast run: {forecast_run}")
print(f"Station: {station_id_to_plot}")
print(f"Models to compare: {list(model_paths.keys())}")

✓ Found local model: cl_m-tft_out-48_freq-1h_wind_05546.pt

Forecast run: 2025-10-25 15:00:00+00:00
Station: 05546
Models to compare: ['100', '50', '50_nostatic', 'local']


### 5.2 Helper Functions

In [7]:
def load_model_and_predict(model_path, config_path, station_id, is_nostatic=False, is_local=False):
    """
    Load a model and make predictions for a specific station for ALL test timestamps.

    The scaler is fitted differently depending on model type:
    - Global models (100, 50): Scaler fitted on ALL relevant stations
    - Local models: Scaler fitted ONLY on the specific station's data

    Args:
        model_path: Path to the model file
        config_path: Path to the global config file
        station_id: Station ID to predict for
        is_nostatic: Whether to remove static features
        is_local: Whether this is a local (station-specific) model

    Returns:
        tuple: (forecast_dates, y_true_all, y_pred_all)
            - forecast_dates: numpy array of timestamps (n_forecasts,)
            - y_true_all: numpy array of true values (n_forecasts, 48)
            - y_pred_all: numpy array of predictions (n_forecasts, 48)
    """
    # Load global config
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)

    if is_nostatic:
        config['params']['static_features'] = []

    config['model']['name'] = 'tft'

    # Get hyperparameters
    features = preprocessing.get_features(config=config)

    # Load station-specific config
    station_config_path = Path(f'configs/wind_50/config_wind_{station_id}.yaml')
    if not station_config_path.exists():
        station_config_path = Path(f'configs/wind_100ex50/config_wind_{station_id}.yaml')

    with open(station_config_path, 'r') as f:
        station_config = yaml.safe_load(f)

    # Load data for this station
    data_dir = station_config['data']['path']
    dfs = preprocessing.get_data(
        data_dir=data_dir,
        config=station_config,
        freq=config['data']['freq'],
        features=features
    )

    # Initialize scaler
    global_scaler_x = StandardScaler()
    target_col = config['data']['target_col']
    test_start = pd.Timestamp(config['data']['test_start'], tz='UTC')
    test_end = pd.Timestamp(config['data']['test_end'], tz='UTC')

    if test_end.hour == 0 and test_end.minute == 0 and test_end.second == 0:
        test_end = test_end.replace(hour=23, minute=0, second=0)

    # FIT SCALER: Different approach for local vs global models
    if is_local:
        # LOCAL MODEL: Fit scaler ONLY on this station's data
        print(f"Fitting scaler on station {station_id} only (local model)...")

        for key, df in dfs.items():
            df_temp = df.copy()
            if target_col in df_temp.columns:
                df_temp.drop(target_col, axis=1, inplace=True)

            t_0 = 0 if config['eval']['eval_on_all_test_data'] else config['eval']['t_0']
            df_train, _ = preprocessing.split_data(
                data=df_temp,
                train_frac=config['data']['train_frac'],
                test_start=test_start,
                test_end=test_end,
                t_0=t_0
            )
            global_scaler_x.partial_fit(df_train)
            del df_temp, df_train

        print(f"Scaler fitted on station {station_id}")

    else:
        # GLOBAL MODEL: Fit scaler on ALL stations
        model_name = model_path.name
        local_configs_dirs = [Path('configs/wind_50'), Path('configs/wind_100ex50')]

        # Find all relevant station configs
        local_config_files = []
        for config_dir in local_configs_dirs:
            if config_dir.exists():
                local_config_files.extend(list(config_dir.glob('config_wind_*.yaml')))
        local_config_files = sorted(local_config_files)

        print(f"Fitting scaler on {len(local_config_files)} stations...")

        for i, local_config_path in enumerate(local_config_files):
            try:
                with open(local_config_path, 'r') as f:
                    local_config = yaml.safe_load(f)

                data_dir_temp = local_config['data']['path']
                dfs_temp = preprocessing.get_data(
                    data_dir=data_dir_temp,
                    config=local_config,
                    freq=config['data']['freq'],
                    features=features
                )

                if len(dfs_temp) == 0:
                    continue

                for key, df in dfs_temp.items():
                    df_temp = df.copy()
                    if target_col in df_temp.columns:
                        df_temp.drop(target_col, axis=1, inplace=True)

                    t_0 = 0 if config['eval']['eval_on_all_test_data'] else config['eval']['t_0']
                    df_train, _ = preprocessing.split_data(
                        data=df_temp,
                        train_frac=config['data']['train_frac'],
                        test_start=test_start,
                        test_end=test_end,
                        t_0=t_0
                    )
                    global_scaler_x.partial_fit(df_train)
                    del df_temp, df_train

                del dfs_temp
                gc.collect()

                if (i + 1) % 20 == 0:
                    print(f"  Fitted scaler on {i+1}/{len(local_config_files)} stations")
            except Exception as e:
                print(f"  Warning: Could not fit scaler on station {local_config_path.stem}: {e}")
                continue

        print(f"Scaler fitted on all {len(local_config_files)} stations")

    # Prepare data with scaler (same for both local and global)
    data_generator = tools.create_data_generator(dfs, config, features, scaler_x=global_scaler_x)
    X_train, y_train, X_test, y_test, test_data = tools.combine_datasets_efficiently(data_generator)

    # Extract feature dimensions
    feature_dims = {}
    if isinstance(X_train, dict):
        feature_dims['observed_dim'] = X_train['observed'].shape[-1]
        feature_dims['known_dim'] = X_train['known'].shape[-1] if 'known' in X_train else 0
        feature_dims['static_dim'] = X_train['static'].shape[-1] if 'static' in X_train else 0
    else:
        feature_dims['observed_dim'] = X_train.shape[-1]
        feature_dims['known_dim'] = 0
        feature_dims['static_dim'] = 0

    # Create station-specific config for model
    import copy
    station_config_model = copy.deepcopy(config)
    station_config_model['model']['feature_dim'] = feature_dims

    # Extract forecast dates
    forecast_dates = None
    for key, data_tuple in test_data.items():
        index_test = data_tuple[2]
        if forecast_dates is None:
            forecast_dates = index_test
        else:
            forecast_dates = np.concatenate([forecast_dates, index_test])

    # Load model
    checkpoint = torch.load(model_path, map_location='cpu')

    if 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    elif 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint

    cleaned_state_dict = {}
    for key, value in state_dict.items():
        if key.startswith('_orig_mod.'):
            cleaned_state_dict[key.replace('_orig_mod.', '')] = value
        else:
            cleaned_state_dict[key] = value

    # Get hyperparameters from model filename
    model_name = model_path.name
    parts = model_name.replace('.pt', '').split('_')
    output_dim = int([p for p in parts if 'out-' in p][0].replace('out-', ''))
    freq_part = [p for p in parts if 'freq-' in p][0]
    freq = freq_part.replace('freq-', '')
    freq_idx = parts.index(freq_part)
    study_name_suffix = '_'.join(parts[freq_idx+1:])
    study_name = f'cl_m-{station_config_model["model"]["name"]}_out-{output_dim}_freq-{freq}_{study_name_suffix}'

    study = None
    try:
        if station_config_model['model']['lookup_hpo']:
            study = hpo.load_study(station_config_model['hpo']['studies_path'], study_name)
    except:
        pass

    hyperparameters = hpo.get_hyperparameters(config=station_config_model, study=study)
    hyperparameters['model_type'] = 'tft'
    hyperparameters.update(feature_dims)

    model = get_model(station_config_model, hyperparameters)
    model.load_state_dict(cleaned_state_dict)
    model.eval()

    # Run inference
    test_observed = X_test['observed'] if isinstance(X_test, dict) else X_test
    test_known = X_test.get('known', None) if isinstance(X_test, dict) else None
    test_static = X_test.get('static', None) if isinstance(X_test, dict) else None

    if not isinstance(test_observed, torch.Tensor):
        test_observed = torch.FloatTensor(test_observed)
    if test_known is not None and not isinstance(test_known, torch.Tensor):
        test_known = torch.FloatTensor(test_known)
    if test_static is not None and not isinstance(test_static, torch.Tensor):
        test_static = torch.FloatTensor(test_static)

    with torch.no_grad():
        if test_static is not None:
            predictions = model(test_observed, test_known, test_static)
        else:
            predictions = model(test_observed, test_known)

    predictions_np = predictions.cpu().numpy()
    y_test_np = y_test if isinstance(y_test, np.ndarray) else y_test.cpu().numpy()

    # Reshape
    predictions_np = predictions_np.reshape(-1, y_test_np.shape[-1])
    predictions_np[predictions_np < 0] = 0

    print(f"  Total forecasts: {len(forecast_dates)}")

    # Clean up
    del model, X_train, y_train, X_test, y_test, dfs
    gc.collect()

    return forecast_dates, y_test_np, predictions_np

### 5.3 Load Predictions for All Three Models

In [8]:
def fit_global_scaler(config_path, model_name, is_nostatic=False):  # ← ADD is_nostatic parameter
    """
    Fit a StandardScaler on the stations relevant for this model.
    - 100-park models: All 100 stations (wind_50 + wind_100ex50)
    - 50-park models: Only 50 stations (wind_50)
    Args:
        config_path: Path to global config
        model_name: Name of model ('100', '50', '50_nostatic', etc.)
        is_nostatic: If True, exclude static features from scaler fitting
    Returns:
        StandardScaler: Fitted scaler
    """
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    # ← CRITICAL FIX: Remove static features if nostatic
    if is_nostatic:
        config['params']['static_features'] = []
    features = preprocessing.get_features(config=config)

    # Determine which configs to use based on model name
    if '100' in model_name:
        # 100-park model: use ALL stations
        local_configs_dirs = [Path('configs/wind_50'), Path('configs/wind_100ex50')]
    elif '50' in model_name:
        # 50-park model: use ONLY wind_50 stations
        local_configs_dirs = [Path('configs/wind_50')]
    else:
        # Default: use all
        local_configs_dirs = [Path('configs/wind_50'), Path('configs/wind_100ex50')]

    # Find all station configs
    local_config_files = []
    for config_dir in local_configs_dirs:
        if config_dir.exists():
            local_config_files.extend(list(config_dir.glob('config_wind_*.yaml')))
    local_config_files = sorted(local_config_files)

    print(f"  Fitting scaler on {len(local_config_files)} stations (relevant for {model_name})...")
    global_scaler_x = StandardScaler()
    target_col = config['data']['target_col']
    test_start = pd.Timestamp(config['data']['test_start'], tz='UTC')
    test_end = pd.Timestamp(config['data']['test_end'], tz='UTC')

    if test_end.hour == 0 and test_end.minute == 0 and test_end.second == 0:
        test_end = test_end.replace(hour=23, minute=0, second=0)

    for i, local_config_path in enumerate(local_config_files):
        try:
            with open(local_config_path, 'r') as f:
                local_config = yaml.safe_load(f)

            data_dir = local_config['data']['path']
            dfs_temp = preprocessing.get_data(
                data_dir=data_dir,
                config=local_config,
                freq=config['data']['freq'],
                features=features
            )

            if len(dfs_temp) == 0:
                continue

            for key, df in dfs_temp.items():
                df_temp = df.copy()
                if target_col in df_temp.columns:
                    df_temp.drop(target_col, axis=1, inplace=True)

                t_0 = 0 if config['eval']['eval_on_all_test_data'] else config['eval']['t_0']
                df_train, _ = preprocessing.split_data(
                    data=df_temp,
                    train_frac=config['data']['train_frac'],
                    test_start=test_start,
                    test_end=test_end,
                    t_0=t_0
                )
                global_scaler_x.partial_fit(df_train)
                del df_temp, df_train

            del dfs_temp
            gc.collect()

            if (i + 1) % 20 == 0:
                print(f"    Fitted scaler on {i+1}/{len(local_config_files)} stations")
        except Exception as e:
            print(f"    Warning: Could not fit scaler on {local_config_path.stem}: {e}")
            continue

    print(f"  ✓ Scaler fitted on {len(local_config_files)} stations")
    return global_scaler_x


def predict_with_fitted_scaler(model_path, config_path, station_id, scaler_x, is_nostatic=False):
    """
    Make predictions using an already fitted scaler.
    Use this for global models to avoid re-fitting the scaler for each station.

    Args:
        model_path: Path to model file
        config_path: Path to config file
        station_id: Station ID to predict for
        scaler_x: Already fitted StandardScaler
        is_nostatic: Whether to remove static features

    Returns:
        tuple: (forecast_dates, y_true_all, y_pred_all)
    """
    # Load config
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)

    if is_nostatic:
        config['params']['static_features'] = []

    config['model']['name'] = 'tft'
    features = preprocessing.get_features(config=config)

    # Load station-specific config
    station_config_path = Path(f'configs/wind_50/config_wind_{station_id}.yaml')
    if not station_config_path.exists():
        station_config_path = Path(f'configs/wind_100ex50/config_wind_{station_id}.yaml')

    with open(station_config_path, 'r') as f:
        station_config = yaml.safe_load(f)

    # Load data for this station
    data_dir = station_config['data']['path']
    dfs = preprocessing.get_data(
        data_dir=data_dir,
        config=station_config,
        freq=config['data']['freq'],
        features=features
    )

    # Prepare data with PROVIDED scaler
    data_generator = tools.create_data_generator(dfs, config, features, scaler_x=scaler_x)
    X_train, y_train, X_test, y_test, test_data = tools.combine_datasets_efficiently(data_generator)

    # Extract feature dimensions
    feature_dims = {}
    if isinstance(X_train, dict):
        feature_dims['observed_dim'] = X_train['observed'].shape[-1]
        feature_dims['known_dim'] = X_train['known'].shape[-1] if 'known' in X_train else 0
        feature_dims['static_dim'] = X_train['static'].shape[-1] if 'static' in X_train else 0
    else:
        feature_dims['observed_dim'] = X_train.shape[-1]
        feature_dims['known_dim'] = 0
        feature_dims['static_dim'] = 0

    # Create config for model
    station_config_model = copy.deepcopy(config)
    station_config_model['model']['feature_dim'] = feature_dims

    # Extract forecast dates
    forecast_dates = None
    for key, data_tuple in test_data.items():
        index_test = data_tuple[2]
        if forecast_dates is None:
            forecast_dates = index_test
        else:
            forecast_dates = np.concatenate([forecast_dates, index_test])

    # Load model
    checkpoint = torch.load(model_path, map_location='cpu')

    if 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    elif 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint

    cleaned_state_dict = {}
    for key, value in state_dict.items():
        if key.startswith('_orig_mod.'):
            cleaned_state_dict[key.replace('_orig_mod.', '')] = value
        else:
            cleaned_state_dict[key] = value

    # Get hyperparameters from model filename
    model_name = model_path.name
    parts = model_name.replace('.pt', '').split('_')
    output_dim = int([p for p in parts if 'out-' in p][0].replace('out-', ''))
    freq_part = [p for p in parts if 'freq-' in p][0]
    freq = freq_part.replace('freq-', '')
    freq_idx = parts.index(freq_part)
    study_name_suffix = '_'.join(parts[freq_idx+1:])
    study_name = f'cl_m-{station_config_model["model"]["name"]}_out-{output_dim}_freq-{freq}_{study_name_suffix}'

    study = None
    try:
        if station_config_model['model']['lookup_hpo']:
            study = hpo.load_study(station_config_model['hpo']['studies_path'], study_name)
    except:
        pass

    hyperparameters = hpo.get_hyperparameters(config=station_config_model, study=study)
    hyperparameters['model_type'] = 'tft'
    hyperparameters.update(feature_dims)

    model = get_model(station_config_model, hyperparameters)
    model.load_state_dict(cleaned_state_dict)
    model.eval()

    # Run inference
    test_observed = X_test['observed'] if isinstance(X_test, dict) else X_test
    test_known = X_test.get('known', None) if isinstance(X_test, dict) else None
    test_static = X_test.get('static', None) if isinstance(X_test, dict) else None

    if not isinstance(test_observed, torch.Tensor):
        test_observed = torch.FloatTensor(test_observed)
    if test_known is not None and not isinstance(test_known, torch.Tensor):
        test_known = torch.FloatTensor(test_known)
    if test_static is not None and not isinstance(test_static, torch.Tensor):
        test_static = torch.FloatTensor(test_static)

    with torch.no_grad():
        if test_static is not None:
            predictions = model(test_observed, test_known, test_static)
        else:
            predictions = model(test_observed, test_known)

    predictions_np = predictions.cpu().numpy()
    y_test_np = y_test if isinstance(y_test, np.ndarray) else y_test.cpu().numpy()

    # Reshape
    predictions_np = predictions_np.reshape(-1, y_test_np.shape[-1])
    predictions_np[predictions_np < 0] = 0

    # Clean up
    del model, X_train, y_train, X_test, y_test, dfs
    gc.collect()

    return forecast_dates, y_test_np, predictions_np

In [11]:
# Try to load existing results
pkl_path = Path('data/test_results/model_comparison.pkl')
if pkl_path.exists():
    print(f"Loading existing results from {pkl_path}...")
    with open(pkl_path, 'rb') as f:
        results_all_stations = pickle.load(f)
    print(f"Loaded results for {len(results_all_stations)} models:")
    for model_name in results_all_stations.keys():
        print(f"  ✓ {model_name}: {len(results_all_stations[model_name])} stations")
    print()
else:
    results_all_stations = {}
    print("No existing results found, starting from scratch.\n")

# Load predictions for each model
for model_name, model_path in model_paths.items():
    # Skip if already computed
    if model_name in results_all_stations and len(results_all_stations[model_name]) > 0:
        print(f"⏭️  Skipping model '{model_name}' (already computed)\n")
        continue

    is_local = (model_name == 'local')

    # For local models, we don't need a global model_path
    if not is_local and model_path is None:
        print(f"⚠ Skipping model '{model_name}' (model file not found)\n")
        continue

    print(f"{'='*80}")
    print(f"MODEL: {model_name}")
    print(f"{'='*80}")

    is_nostatic = 'nostatic' in model_name
    results_all_stations[model_name] = {}

    # Determine which stations to test based on model type
    # LOCAL models: always use wind_50 stations only
    if is_local:
        local_configs_dirs = [Path('configs/wind_50')]
        print(f"Local model: using wind_50 stations")
    elif '100' in model_name:
        local_configs_dirs = [Path('configs/wind_50'), Path('configs/wind_100ex50')]
        print(f"100-park model: using all 100 stations")
    elif '50' in model_name:
        local_configs_dirs = [Path('configs/wind_50')]
        print(f"50-park model: using 50 stations")
    else:
        local_configs_dirs = [Path('configs/wind_50'), Path('configs/wind_100ex50')]
        print(f"Default: using all 100 stations")

    # Get station list for THIS model
    local_config_files = []
    for config_dir in local_configs_dirs:
        if config_dir.exists():
            local_config_files.extend(list(config_dir.glob('config_wind_*.yaml')))
    local_config_files = sorted(local_config_files)
    station_ids_for_model = [f.stem.replace('config_wind_', '') for f in local_config_files]

    print(f"Testing on {len(station_ids_for_model)} stations\n")

    # GLOBAL MODELS: Fit scaler ONCE, then predict for all stations
    if not is_local:
        print(f"Global model - fitting scaler on relevant stations...")
        global_scaler = fit_global_scaler(config_path, model_name, is_nostatic=is_nostatic)
        print()

        for i, station_id in enumerate(station_ids_for_model):
            print(f"[{i+1}/{len(station_ids_for_model)}] Station {station_id}...", end=' ', flush=True)

            try:
                forecast_dates, y_true_all, y_pred_all = predict_with_fitted_scaler(
                    model_path=model_path,
                    config_path=config_path,
                    station_id=station_id,
                    scaler_x=global_scaler,
                    is_nostatic=is_nostatic
                )

                results_all_stations[model_name][station_id] = {
                    'y_true_all': y_true_all,
                    'y_pred_all': y_pred_all,
                    'forecast_dates': forecast_dates
                }

                print(f"✓ ({y_true_all.shape[0]} forecasts)")

            except Exception as e:
                print(f"✗ Error: {e}")
                continue

    # LOCAL MODELS: Each station has its own model and scaler
else:
    print(f"Local model - using station-specific models and scalers\n")
    for i, station_id in enumerate(station_ids_for_model):
        print(f"[{i+1}/{len(station_ids_for_model)}] Station {station_id}...", end=' ', flush=True)
        try:
            # Find station-specific model
            local_model_pattern = f'cl_m-tft_out-48_freq-1h_wind_{station_id}.pt'
            station_model_path = Path('models') / local_model_pattern
            if not station_model_path.exists():
                print(f"✗ Model not found")
                continue
            # ← CRITICAL FIX: Use station-specific config for local models!
            station_config_path = Path(f'configs/wind_50/config_wind_{station_id}.yaml')

            if not station_config_path.exists():
                print(f"✗ Config not found")
                continue
            forecast_dates, y_true_all, y_pred_all = load_model_and_predict(
                model_path=station_model_path,
                config_path=station_config_path,  # ← Use station config, not global!
                station_id=station_id,
                is_nostatic=False,  # ← Doesn't matter, config already has empty static_features
                is_local=True
            )
            results_all_stations[model_name][station_id] = {
                'y_true_all': y_true_all,
                'y_pred_all': y_pred_all,
                'forecast_dates': forecast_dates
            }
            print(f"✓ ({y_true_all.shape[0]} forecasts)")
        except Exception as e:
            print(f"✗ Error: {e}")
            continue

    # Save after each model completes
    print(f"\n💾 Saving progress to {pkl_path}...")
    pkl_path.parent.mkdir(parents=True, exist_ok=True)
    with open(pkl_path, 'wb') as f:
        pickle.dump(results_all_stations, f)
    print(f"✓ Saved!\n")

print("="*80)
print("ALL MODELS AND STATIONS LOADED!")
print("="*80)
for model_name in results_all_stations.keys():
    print(f"  {model_name}: {len(results_all_stations[model_name])} stations")

Loading existing results from data/test_results/model_comparison.pkl...
Loaded results for 4 models:
  ✓ 100: 100 stations
  ✓ 50: 50 stations
  ✓ 50_nostatic: 50 stations
  ✓ local: 50 stations

⏭️  Skipping model '100' (already computed)

⏭️  Skipping model '50' (already computed)

⏭️  Skipping model '50_nostatic' (already computed)

⏭️  Skipping model 'local' (already computed)

Local model - using station-specific models and scalers



NameError: name 'station_ids_for_model' is not defined

In [12]:
# ==========================================
# DEFINE FORECAST RUN AND STATION HERE
# ==========================================
forecast_run = '2025-10-25 15:00:00+00:00'  # ← CHANGE THIS
station_id_to_plot = '05546'  # ← CHANGE THIS
# ==========================================

# Filter for specific station and forecast run
results = {}

for model_name in results_all_stations.keys():
    if station_id_to_plot not in results_all_stations[model_name]:
        print(f"Warning: Station {station_id_to_plot} not found for model {model_name}")
        continue

    station_data = results_all_stations[model_name][station_id_to_plot]
    forecast_dates = station_data['forecast_dates']
    y_true_all = station_data['y_true_all']
    y_pred_all = station_data['y_pred_all']

    # Find index for the specific forecast timestamp
    forecast_ts = pd.Timestamp(forecast_run)
    try:
        forecast_idx = np.where(forecast_dates == forecast_ts)[0][0]
    except (IndexError, TypeError):
        # Try string comparison if timestamp comparison fails
        forecast_dates_pd = pd.DatetimeIndex(forecast_dates)
        forecast_idx = forecast_dates_pd.get_loc(forecast_ts)

    # Extract the specific forecast
    y_true_single = y_true_all[forecast_idx, :]
    y_pred_single = y_pred_all[forecast_idx, :]

    results[model_name] = {
        'y_true': y_true_single,
        'y_pred': y_pred_single
    }

    print(f"{model_name}: Station {station_id_to_plot}, Forecast {forecast_dates[forecast_idx]}")

print(f"\n✓ Ready to plot: Station {station_id_to_plot}, Forecast run {forecast_run}")

100: Station 05546, Forecast 2025-10-25 15:00:00+00:00

✓ Ready to plot: Station 05546, Forecast run 2025-10-25 15:00:00+00:00


### 5.4 Create Interactive Plotly Visualization

In [ ]:
# Create plotly figure
fig = go.Figure()

# X-axis: Hours (0 to 47)
hours = np.arange(48)

# Get y_true (same for all models) - use first available model
y_true = None
for model_name in results.keys():
    if 'y_true' in results[model_name]:
        y_true = results[model_name]['y_true']
        break

if y_true is None:
    raise ValueError("No y_true data found in results!")

# Add ground truth line
fig.add_trace(go.Scatter(
    x=hours,
    y=y_true,
    mode='lines',
    name='Ground Truth',
    line=dict(color='blue', width=3),
    hovertemplate='<b>True</b><br>Hour: %{x}<br>Power: %{y:.3f}<extra></extra>'
))

# Define colors for predictions - Kräftigere Farben für bessere Sichtbarkeit
model_colors = {
    '100': 'rgb(34, 139, 34)',       # Forest Green (kräftiges Grün)
    '50': 'rgb(255, 140, 0)',        # Dark Orange (kräftiges Orange)
    '50_nostatic': 'rgb(220, 20, 60)',  # Crimson (kräftiges Rot)
    'local': 'rgb(138, 43, 226)'     # Blue Violet (kräftiges Lila)
}

# Calculate metrics for each model and create table text
metrics_text = '<b>Metrics (%)<br><br>'
metrics_text += 'Model         R²     RMSE    MAE<br>'
metrics_text += '────────────────────────────────<br>'

for model_name, data in results.items():
    y_pred = data['y_pred']

    r2 = r2_score(y_true, y_pred) * 100  # Convert to %
    rmse = np.sqrt(mean_squared_error(y_true, y_pred)) * 100  # Convert to %
    mae = mean_absolute_error(y_true, y_pred) * 100  # Convert to %

    # Format model name to fixed width
    model_display = f'{model_name:<13}'
    metrics_text += f'{model_display} {r2:5.1f}  {rmse:5.1f}  {mae:5.1f}<br>'

    # Get color for this model (use default if not defined)
    color = model_colors.get(model_name, 'rgb(100, 100, 100)')  # Default gray

    # Add prediction lines
    fig.add_trace(go.Scatter(
        x=hours,
        y=y_pred,
        mode='lines',
        name=f'Model {model_name}',
        line=dict(color=color, width=3, dash='dash'),
        hovertemplate=f'<b>Model {model_name}</b><br>Hour: %{{x}}<br>Power: %{{y:.3f}}<extra></extra>'
    ))

metrics_text += '</b>'

# Update layout
fig.update_layout(
    title=f'Forecast Comparison for Station {station_id_to_plot}<br>{forecast_run}',
    xaxis=dict(
        title=dict(
            text='<b>Forecast Horizon (Hours)</b>',
            font=dict(size=16, color='black', family='Arial, sans-serif')
        ),
        tickfont=dict(size=14)
    ),
    yaxis=dict(
        title=dict(
            text='<b>Normalized Power</b>',
            font=dict(size=16, color='black', family='Arial, sans-serif')
        ),
        tickfont=dict(size=14)
    ),
    hovermode='x unified',
    template='plotly_white',
    width=1200,
    height=600,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.8)",
        font=dict(size=16)
    )
)

# Add metrics table as annotation
fig.add_annotation(
    text=metrics_text,
    xref="paper",
    yref="paper",
    x=0.99,  # Right edge
    y=0.99,  # Top edge
    xanchor="right", # 'right'
    yanchor="top", # 'bottom'
    showarrow=False,
    bgcolor="rgba(255, 255, 255, 0.95)",
    bordercolor="black",
    borderwidth=1,
    borderpad=10,
    align="left",
    font=dict(
        family="Courier New, monospace",
        size=14,
        color="black"
    )
)

# Save plot
os.makedirs('plots/comparisons', exist_ok=True)
fig.write_html(f'plots/comparisons/forecast_comparison_{station_id_to_plot}_{forecast_run.replace(":", "-").replace(" ", "_")}.html')
# Show the plot
fig.show()

### 5.5 Calculate Metrics for this Forecast Run

In [28]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

print(f"Metrics for Forecast Run: {forecast_run}, Station: {station_id_to_plot}\n")
print(f"{'Model':<20} {'R²':<10} {'RMSE':<10} {'MAE':<10}")
print("="*50)

for model_name, data in results.items():
    y_true = data['y_true']
    y_pred = data['y_pred']

    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    print(f"{model_name:<20} {r2:<10.4f} {rmse:<10.4f} {mae:<10.4f}")

Metrics for Forecast Run: 2025-10-25 15:00:00+00:00, Station: 05546

Model                R²         RMSE       MAE       
100                  0.9291     0.0747     0.0564    
50                   0.8422     0.1114     0.0715    
50_nostatic          0.4366     0.2105     0.1486    
